# Cantonese data scaling: LoRA r=8 vs full fine-tuning

This self-contained Colab compares **LoRA r=8 + parsing-layer fine-tuning** with **full Transformer + parsing-layer fine-tuning** at four nested Cantonese training sizes: approximately 10%, 25%, 50%, and 100% of the fixed 803-sentence train split. Select an A100 GPU and choose **Runtime → Run all**.

The custom 101-sentence dev and 100-sentence test sets remain exactly fixed. Duplicate-text groups are never divided, and training subsets are deterministic, nested, and greedily balanced across the four official source domains. Each condition starts from the same Stanza 1.14 Mandarin ELECTRA-large parser, uses the same frozen predicted POS/lemma inputs, and expands the output vocabulary only with Cantonese relations observed in that condition's training subset.

Training is controlled by epochs rather than a fixed number of steps: 120 maximum epochs, evaluation every 5 epochs, and early stopping after 40 epochs without DEV LAS improvement. This prevents smaller subsets from receiving many more passes over their data. DEV selects each checkpoint; the predeclared fixed TEST is evaluated once afterward for every condition.

The notebook mounts Google Drive and writes a recovery checkpoint after every DEV evaluation (every 5 epochs), including model, optimizer, scheduler, RNG states, and progress. If Colab disconnects, reconnect and Run all again: verified completed conditions are skipped, and the interrupted condition resumes from its latest completed evaluation. Full-FT recovery files are large and make training somewhat slower, but they prevent a long condition from being lost. Recovery checkpoints are deleted after that condition's DEV and TEST predictions and result record are safely written.

Important limitation: this custom test was previously used to select the Mandarin model family, so it is fixed and comparable but not untouched. The default experiment uses one seed; small method differences require later multi-seed confirmation.


In [ ]:
# Pinned Python packages.  Restart the runtime after this cell if Colab asks.
%pip install -q stanza==1.14.0 transformers==4.56.2 peft==0.17.1 huggingface-hub==0.34.4


In [ ]:
from pathlib import Path
import os, json, hashlib, random, shutil, gc, copy, importlib.util, logging
import numpy as np
import torch

SEED = 42
WORK = Path('/content/yue_data_scaling')
DATA = WORK / 'data'
MODELS = WORK / 'stanza_resources_1.14.0'
OUT = WORK / 'outputs'
HF_HOME = WORK / 'hf_home'
for p in (DATA, MODELS, OUT, HF_HOME): p.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME)

RAW_URL = 'https://raw.githubusercontent.com/UniversalDependencies/UD_Cantonese-HK/r2.18/yue_hk-ud-test.conllu'
RAW_SHA256 = 'cbd843a195d0db4cdafbf6fcafb7b7b559afea750411006f4728311e70cc4e2a'
SPLIT_POSITIONS = {'train': [1, 2, 3, 5, 6, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 27, 29, 30, 32, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 48, 49, 51, 52, 53, 54, 55, 56, 57, 58, 60, 61, 62, 63, 64, 65, 66, 68, 70, 71, 72, 74, 76, 77, 78, 79, 80, 81, 84, 85, 86, 87, 88, 89, 90, 92, 93, 94, 98, 99, 100, 102, 103, 104, 107, 108, 109, 110, 111, 112, 113, 115, 116, 118, 119, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 150, 151, 152, 154, 155, 156, 157, 158, 159, 160, 161, 162, 164, 165, 167, 169, 171, 172, 175, 176, 177, 178, 179, 180, 181, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 205, 206, 207, 208, 209, 210, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 223, 226, 227, 229, 230, 234, 235, 237, 238, 239, 240, 243, 244, 246, 248, 249, 251, 252, 253, 254, 255, 256, 257, 258, 259, 261, 263, 264, 265, 266, 267, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 284, 287, 289, 290, 291, 292, 295, 296, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 310, 311, 312, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 339, 340, 341, 342, 343, 344, 345, 346, 347, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 362, 363, 364, 365, 366, 367, 368, 370, 371, 372, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 386, 387, 388, 390, 391, 393, 394, 396, 397, 398, 399, 400, 402, 403, 404, 405, 406, 407, 408, 413, 414, 415, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 428, 429, 431, 433, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 454, 455, 456, 458, 459, 460, 461, 462, 464, 465, 466, 467, 468, 469, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 485, 486, 487, 488, 490, 491, 492, 493, 496, 497, 499, 500, 501, 502, 503, 505, 506, 507, 508, 509, 510, 511, 512, 513, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 524, 525, 526, 527, 528, 529, 531, 532, 533, 534, 535, 537, 538, 539, 540, 541, 543, 544, 545, 547, 548, 549, 550, 551, 552, 553, 554, 555, 556, 557, 558, 559, 560, 561, 562, 563, 564, 565, 566, 568, 569, 570, 571, 572, 573, 575, 576, 578, 580, 581, 582, 583, 584, 585, 587, 588, 589, 590, 591, 592, 593, 596, 597, 598, 600, 601, 602, 603, 605, 606, 607, 608, 609, 611, 612, 613, 614, 615, 616, 617, 618, 619, 620, 622, 623, 624, 625, 626, 629, 630, 631, 632, 635, 636, 637, 638, 639, 641, 642, 643, 644, 645, 647, 649, 650, 651, 652, 654, 655, 656, 657, 658, 659, 660, 661, 662, 664, 665, 666, 667, 668, 669, 670, 671, 674, 675, 676, 677, 678, 679, 681, 682, 683, 687, 689, 690, 691, 692, 694, 696, 697, 698, 699, 700, 702, 703, 704, 705, 706, 708, 709, 710, 711, 712, 713, 714, 715, 717, 718, 719, 720, 721, 723, 724, 725, 727, 730, 732, 733, 735, 736, 737, 738, 739, 740, 741, 742, 743, 745, 746, 747, 749, 750, 752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 762, 764, 767, 770, 771, 772, 773, 774, 775, 778, 779, 780, 781, 782, 783, 785, 786, 787, 790, 791, 793, 795, 796, 797, 799, 800, 801, 802, 804, 805, 806, 808, 809, 810, 812, 813, 814, 815, 818, 819, 820, 822, 823, 825, 826, 827, 828, 829, 830, 831, 832, 833, 834, 836, 838, 839, 840, 841, 842, 843, 844, 845, 846, 847, 848, 849, 850, 851, 852, 853, 857, 859, 860, 861, 862, 863, 864, 865, 866, 867, 868, 869, 870, 872, 873, 874, 875, 876, 877, 878, 880, 881, 882, 883, 884, 886, 888, 889, 890, 891, 892, 893, 894, 895, 896, 897, 899, 900, 901, 902, 903, 904, 907, 908, 909, 910, 911, 912, 915, 916, 917, 919, 920, 922, 923, 924, 925, 926, 927, 928, 929, 930, 933, 934, 935, 937, 938, 939, 940, 942, 943, 945, 946, 947, 948, 949, 950, 951, 953, 954, 955, 956, 957, 958, 962, 964, 965, 966, 967, 968, 970, 971, 972, 973, 974, 976, 977, 978, 980, 981, 982, 984, 985, 986, 989, 990, 992, 993, 994, 996, 998, 1000, 1001, 1002, 1004], 'dev': [4, 12, 50, 59, 67, 69, 75, 95, 114, 120, 148, 153, 163, 168, 174, 182, 222, 224, 225, 232, 242, 262, 268, 281, 283, 285, 286, 288, 337, 338, 348, 410, 412, 416, 427, 430, 432, 434, 453, 463, 470, 494, 498, 504, 530, 536, 546, 567, 574, 579, 595, 599, 610, 627, 628, 640, 653, 672, 685, 686, 688, 693, 701, 716, 726, 728, 729, 731, 734, 765, 766, 768, 769, 776, 792, 794, 798, 803, 816, 817, 824, 835, 837, 854, 871, 887, 905, 914, 931, 932, 936, 941, 960, 961, 963, 975, 979, 987, 997, 999, 1003], 'test': [7, 26, 28, 31, 33, 45, 47, 73, 82, 83, 91, 96, 97, 101, 105, 106, 117, 132, 149, 166, 170, 173, 204, 211, 228, 231, 233, 236, 241, 245, 247, 250, 260, 282, 293, 294, 297, 309, 313, 361, 369, 373, 385, 389, 392, 395, 401, 409, 411, 452, 457, 484, 489, 495, 542, 577, 586, 594, 604, 621, 633, 634, 646, 648, 663, 673, 680, 684, 695, 707, 722, 744, 748, 751, 763, 777, 784, 788, 789, 807, 811, 821, 855, 856, 858, 879, 885, 898, 906, 913, 918, 921, 944, 952, 959, 969, 983, 988, 991, 995]}
SPLIT_SHA256 = {'train': 'b2d6b96af234f22825bb007a9a2a57ef49619c59747dc582dca7fdb1a4331a2d', 'dev': '41bc28d903457e70a4747307e56b3eb7820f2e6d3ede10ab549fd4a85aef63b7', 'test': '1d7b19ac4c0a75d58a80482413ff9cac63ed18917262c18fa1f92fc0f871c0a0'}
EVAL_URL = 'https://universaldependencies.org/conll18/conll18_ud_eval.py'
EVAL_SHA256 = '1072e02af00b1a56205b5e8216d51dee9b8944a104d80744afaccc78859fcb16'
HF_REPO = 'hfl/chinese-electra-180g-large-discriminator'
# If the earlier 41.32% run recorded a HF commit, paste it here.  None resolves HEAD once,
# records the exact commit, and then forces offline reuse for the rest of this run.
HF_ELECTRA_REVISION = 'd017e219578df8e4885484edbc8969dbdea9cbe0'

SUBSETS = {'p010': {'proportion': 0.1, 'target_sentence_count': 80, 'sentence_count': 80, 'group_count': 80, 'positions': [8, 38, 40, 54, 58, 60, 74, 90, 99, 107, 108, 116, 128, 169, 189, 194, 195, 220, 227, 229, 244, 253, 273, 278, 290, 298, 316, 341, 343, 365, 375, 383, 403, 415, 419, 425, 426, 436, 480, 497, 499, 514, 540, 547, 548, 550, 559, 561, 615, 620, 624, 629, 645, 667, 674, 698, 705, 720, 741, 752, 759, 770, 772, 780, 786, 806, 812, 827, 830, 850, 882, 883, 886, 894, 899, 904, 927, 943, 953, 981], 'gold_sha256': '550bf5d9d21e5ac4649c1c5c4315570086f103b85c374727f2d82b637c632612', 'source_distribution': {'Election of President (LegCo, 2016-10-12)': 27, 'Missing days / 小時光': 33, 'Tempo in Temple / 廟眾樂樂': 11, 'What day is today / 今日星期幾': 9}}, 'p025': {'proportion': 0.25, 'target_sentence_count': 201, 'sentence_count': 201, 'group_count': 201, 'positions': [8, 11, 25, 30, 32, 38, 40, 41, 54, 55, 57, 58, 60, 65, 74, 77, 85, 88, 90, 92, 98, 99, 100, 107, 108, 116, 119, 128, 129, 136, 158, 169, 175, 179, 189, 194, 195, 198, 202, 207, 209, 220, 227, 229, 230, 240, 244, 248, 252, 253, 270, 273, 278, 279, 287, 290, 296, 298, 304, 305, 316, 317, 320, 330, 331, 339, 340, 341, 343, 351, 354, 356, 357, 365, 375, 377, 383, 386, 391, 399, 403, 405, 406, 415, 418, 419, 422, 425, 426, 428, 436, 437, 443, 444, 450, 459, 462, 480, 485, 486, 497, 499, 500, 501, 502, 514, 522, 528, 529, 539, 540, 547, 548, 550, 551, 552, 559, 561, 573, 597, 598, 606, 613, 614, 615, 618, 620, 624, 625, 629, 630, 636, 645, 651, 661, 664, 666, 667, 674, 687, 698, 705, 710, 719, 720, 741, 749, 752, 759, 760, 761, 762, 770, 772, 780, 781, 785, 786, 805, 806, 812, 814, 819, 820, 825, 827, 828, 830, 846, 850, 853, 872, 874, 877, 882, 883, 886, 889, 892, 894, 897, 899, 904, 908, 911, 919, 923, 924, 925, 927, 929, 940, 943, 951, 953, 966, 971, 981, 985, 989, 992], 'gold_sha256': 'e3ae44fda225a540dae6e0cbd38538b9a0c8eee525cab75d490e13b6bbd5fc4c', 'source_distribution': {'Election of President (LegCo, 2016-10-12)': 68, 'Missing days / 小時光': 83, 'Tempo in Temple / 廟眾樂樂': 29, 'What day is today / 今日星期幾': 21}}, 'p050': {'proportion': 0.5, 'target_sentence_count': 402, 'sentence_count': 402, 'group_count': 402, 'positions': [1, 6, 8, 11, 16, 17, 18, 19, 20, 21, 22, 25, 29, 30, 32, 34, 38, 39, 40, 41, 43, 49, 51, 52, 54, 55, 56, 57, 58, 60, 63, 64, 65, 70, 74, 76, 77, 78, 84, 85, 86, 88, 89, 90, 92, 98, 99, 100, 103, 107, 108, 109, 112, 116, 118, 119, 121, 128, 129, 130, 136, 145, 147, 152, 154, 155, 158, 160, 161, 169, 171, 175, 176, 179, 184, 185, 189, 192, 193, 194, 195, 197, 198, 202, 205, 206, 207, 208, 209, 214, 220, 227, 229, 230, 235, 240, 244, 246, 248, 249, 252, 253, 256, 257, 259, 270, 273, 276, 277, 278, 279, 287, 289, 290, 292, 295, 296, 298, 299, 301, 302, 304, 305, 306, 308, 314, 316, 317, 320, 322, 324, 326, 330, 331, 332, 333, 334, 339, 340, 341, 343, 350, 351, 353, 354, 355, 356, 357, 363, 365, 370, 375, 377, 378, 383, 386, 387, 391, 396, 397, 399, 403, 405, 406, 408, 414, 415, 417, 418, 419, 420, 421, 422, 423, 425, 426, 428, 436, 437, 438, 440, 443, 444, 450, 458, 459, 460, 461, 462, 468, 472, 476, 479, 480, 485, 486, 488, 491, 497, 499, 500, 501, 502, 503, 507, 509, 511, 514, 521, 522, 523, 526, 527, 528, 529, 531, 532, 535, 539, 540, 541, 547, 548, 549, 550, 551, 552, 556, 559, 561, 564, 566, 572, 573, 578, 580, 581, 592, 593, 597, 598, 606, 608, 611, 613, 614, 615, 616, 617, 618, 620, 623, 624, 625, 626, 629, 630, 632, 636, 637, 638, 639, 644, 645, 647, 651, 658, 661, 664, 666, 667, 670, 674, 676, 677, 678, 687, 689, 690, 691, 696, 697, 698, 703, 705, 710, 711, 715, 718, 719, 720, 730, 735, 741, 743, 745, 747, 749, 752, 757, 759, 760, 761, 762, 767, 770, 772, 779, 780, 781, 783, 785, 786, 787, 793, 801, 805, 806, 809, 812, 814, 819, 820, 825, 827, 828, 830, 831, 840, 846, 847, 849, 850, 851, 852, 853, 859, 863, 864, 867, 868, 870, 872, 874, 875, 876, 877, 882, 883, 884, 886, 888, 889, 892, 894, 897, 899, 904, 907, 908, 909, 911, 912, 919, 923, 924, 925, 926, 927, 929, 934, 935, 938, 940, 942, 943, 947, 948, 951, 953, 954, 956, 957, 965, 966, 968, 970, 971, 972, 973, 974, 977, 978, 980, 981, 984, 985, 989, 992, 994, 998, 1004], 'gold_sha256': '0f58a0d6f0fffc39062b1c026f1725da0778dda6dca6b001d5ada173c9e38de2', 'source_distribution': {'Election of President (LegCo, 2016-10-12)': 137, 'Missing days / 小時光': 165, 'Tempo in Temple / 廟眾樂樂': 57, 'What day is today / 今日星期幾': 43}}, 'p100': {'proportion': 1.0, 'target_sentence_count': 803, 'sentence_count': 803, 'group_count': 778, 'positions': [1, 2, 3, 5, 6, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 27, 29, 30, 32, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 48, 49, 51, 52, 53, 54, 55, 56, 57, 58, 60, 61, 62, 63, 64, 65, 66, 68, 70, 71, 72, 74, 76, 77, 78, 79, 80, 81, 84, 85, 86, 87, 88, 89, 90, 92, 93, 94, 98, 99, 100, 102, 103, 104, 107, 108, 109, 110, 111, 112, 113, 115, 116, 118, 119, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 150, 151, 152, 154, 155, 156, 157, 158, 159, 160, 161, 162, 164, 165, 167, 169, 171, 172, 175, 176, 177, 178, 179, 180, 181, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 205, 206, 207, 208, 209, 210, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 223, 226, 227, 229, 230, 234, 235, 237, 238, 239, 240, 243, 244, 246, 248, 249, 251, 252, 253, 254, 255, 256, 257, 258, 259, 261, 263, 264, 265, 266, 267, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 284, 287, 289, 290, 291, 292, 295, 296, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 310, 311, 312, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 339, 340, 341, 342, 343, 344, 345, 346, 347, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 362, 363, 364, 365, 366, 367, 368, 370, 371, 372, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 386, 387, 388, 390, 391, 393, 394, 396, 397, 398, 399, 400, 402, 403, 404, 405, 406, 407, 408, 413, 414, 415, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 428, 429, 431, 433, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 454, 455, 456, 458, 459, 460, 461, 462, 464, 465, 466, 467, 468, 469, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 485, 486, 487, 488, 490, 491, 492, 493, 496, 497, 499, 500, 501, 502, 503, 505, 506, 507, 508, 509, 510, 511, 512, 513, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 524, 525, 526, 527, 528, 529, 531, 532, 533, 534, 535, 537, 538, 539, 540, 541, 543, 544, 545, 547, 548, 549, 550, 551, 552, 553, 554, 555, 556, 557, 558, 559, 560, 561, 562, 563, 564, 565, 566, 568, 569, 570, 571, 572, 573, 575, 576, 578, 580, 581, 582, 583, 584, 585, 587, 588, 589, 590, 591, 592, 593, 596, 597, 598, 600, 601, 602, 603, 605, 606, 607, 608, 609, 611, 612, 613, 614, 615, 616, 617, 618, 619, 620, 622, 623, 624, 625, 626, 629, 630, 631, 632, 635, 636, 637, 638, 639, 641, 642, 643, 644, 645, 647, 649, 650, 651, 652, 654, 655, 656, 657, 658, 659, 660, 661, 662, 664, 665, 666, 667, 668, 669, 670, 671, 674, 675, 676, 677, 678, 679, 681, 682, 683, 687, 689, 690, 691, 692, 694, 696, 697, 698, 699, 700, 702, 703, 704, 705, 706, 708, 709, 710, 711, 712, 713, 714, 715, 717, 718, 719, 720, 721, 723, 724, 725, 727, 730, 732, 733, 735, 736, 737, 738, 739, 740, 741, 742, 743, 745, 746, 747, 749, 750, 752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 762, 764, 767, 770, 771, 772, 773, 774, 775, 778, 779, 780, 781, 782, 783, 785, 786, 787, 790, 791, 793, 795, 796, 797, 799, 800, 801, 802, 804, 805, 806, 808, 809, 810, 812, 813, 814, 815, 818, 819, 820, 822, 823, 825, 826, 827, 828, 829, 830, 831, 832, 833, 834, 836, 838, 839, 840, 841, 842, 843, 844, 845, 846, 847, 848, 849, 850, 851, 852, 853, 857, 859, 860, 861, 862, 863, 864, 865, 866, 867, 868, 869, 870, 872, 873, 874, 875, 876, 877, 878, 880, 881, 882, 883, 884, 886, 888, 889, 890, 891, 892, 893, 894, 895, 896, 897, 899, 900, 901, 902, 903, 904, 907, 908, 909, 910, 911, 912, 915, 916, 917, 919, 920, 922, 923, 924, 925, 926, 927, 928, 929, 930, 933, 934, 935, 937, 938, 939, 940, 942, 943, 945, 946, 947, 948, 949, 950, 951, 953, 954, 955, 956, 957, 958, 962, 964, 965, 966, 967, 968, 970, 971, 972, 973, 974, 976, 977, 978, 980, 981, 982, 984, 985, 986, 989, 990, 992, 993, 994, 996, 998, 1000, 1001, 1002, 1004], 'gold_sha256': 'b2d6b96af234f22825bb007a9a2a57ef49619c59747dc582dca7fdb1a4331a2d', 'source_distribution': {'Election of President (LegCo, 2016-10-12)': 273, 'Missing days / 小時光': 330, 'Tempo in Temple / 廟眾樂樂': 115, 'What day is today / 今日星期幾': 85}}}
METHODS = ['lora_r8', 'full_ft']
CFG = dict(rank=8, alpha=16, dropout=0.10,
           targets=['query', 'value', 'output.dense', 'intermediate.dense'],
           parser_lr=1e-3, transformer_lr=2e-5, batch_size=900,
           max_epochs=120, eval_every_epochs=5, patience_epochs=40,
           max_grad_norm=1.0)
assert list(SUBSETS)==['p010','p025','p050','p100']

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1<<20), b''): h.update(b)
    return h.hexdigest()

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'ELECTRA-large training requires a Colab GPU runtime.'
print('device:', DEVICE, 'config:', CFG)


In [ ]:
# Download the pinned UD source and reconstruct the project's exact custom splits.
import urllib.request
raw_path = DATA / 'yue_hk-ud-test.r2.18.conllu'
urllib.request.urlretrieve(RAW_URL, raw_path)
assert sha256(raw_path) == RAW_SHA256

raw = raw_path.read_text(encoding='utf-8')
blocks = raw.strip().split('\n\n')
assert len(blocks) == 1004
for split, positions in SPLIT_POSITIONS.items():
    # CoNLL-U requires a blank line between sentence blocks.  The project files
    # also end in one blank line, hence the final two newline characters.
    text = '\n\n'.join(blocks[i-1] for i in positions) + '\n\n'
    path = DATA / f'{split}.conllu'
    path.write_text(text, encoding='utf-8')
    assert sha256(path) == SPLIT_SHA256[split], (split, sha256(path))
    print(split, len(positions), sha256(path))

eval_path = WORK / 'conll18_ud_eval.py'
urllib.request.urlretrieve(EVAL_URL, eval_path)
assert sha256(eval_path) == EVAL_SHA256
spec=importlib.util.spec_from_file_location('official_conll18', eval_path)
official=importlib.util.module_from_spec(spec); spec.loader.exec_module(official)

def conll18(path_gold, path_system):
    return official.evaluate(official.load_conllu_file(str(path_gold)),
                             official.load_conllu_file(str(path_system)))


In [ ]:
# Resolve one exact HF commit and download it into an isolated cache.
from huggingface_hub import snapshot_download
snapshot = Path(snapshot_download(HF_REPO, revision=HF_ELECTRA_REVISION, cache_dir=HF_HOME/'hub'))
HF_COMMIT = snapshot.name
assert len(HF_COMMIT) == 40
print('frozen HF revision:', HF_COMMIT)


In [ ]:
# Download the exact Stanza 1.14 Mandarin processors and verify registry/artifact hashes.
import stanza
from stanza.resources.common import download_resources_json, load_resources_json
from stanza.pipeline.core import DownloadMethod
assert stanza.__version__ == '1.14.0', f'Expected Stanza 1.14.0, found {stanza.__version__}. Restart runtime and rerun.'
download_resources_json(model_dir=str(MODELS))
resources_path=MODELS/'resources.json'
assert sha256(resources_path) == '4e41c1df152146fa26ed0c006a08feea7a60bb3414bb6d57dbda24ad2e3cb99c'
packages={'tokenize':'gsdsimp','pos':'gsdsimp_electra-large','lemma':'gsdsimp_charlm','depparse':'gsdsimp_electra-large'}
stanza.download('zh-hans', model_dir=str(MODELS), package=None, processors=packages, verbose=True)
resources=load_resources_json(model_dir=str(MODELS))
expected_md5={'tokenize':'48f993223d568afedc2893f7cd76719c','pos':'73859e5ec15bedc545d6deafd6ddba94','lemma':'b49edd41abb063a87b125ec53aa5b96c','depparse':'c7ea98d93459b22720337a9dcba1a848'}
artifacts={}
for proc,pkg in packages.items():
    p=MODELS/'zh-hans'/proc/f'{pkg}.pt'
    registry=resources['zh-hans'][proc][pkg]['md5']
    assert registry == expected_md5[proc] and p.exists()
    artifacts[proc]={'path':str(p),'registry_md5':registry,'sha256':sha256(p)}
print(json.dumps(artifacts, indent=2))
# Processor artifacts and the exact Transformer snapshot are now present.  Forbid
# later Hub lookups so every POS/parser load reuses this one cached revision.
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'


In [ ]:
# Run frozen POS/lemma exactly once and build pretagged train/dev/test caches.
# Gold sentence boundaries and integer-ID FORM tokenization are fed to the pipeline.
# Only LEMMA/UPOS/XPOS/FEATS are replaced; gold HEAD/DEPREL remain training targets.
tagger = stanza.Pipeline(lang='zh-hans', dir=str(MODELS),
    processors={k:v for k,v in packages.items() if k != 'depparse'},
    tokenize_pretokenized=True, use_gpu=True,
    download_method=DownloadMethod.REUSE_RESOURCES, verbose=False)
for proc in ('pos','lemma'):
    model=tagger.processors[proc]._trainer.model
    model.eval()
    for p in model.parameters(): p.requires_grad=False
    assert not any(p.requires_grad for p in model.parameters()), f'{proc} unexpectedly trainable'

def split_blocks(text): return text.strip().split('\n\n')
def integer_rows(block):
    return [line.split('\t') for line in block.splitlines()
            if line and not line.startswith('#') and line.split('\t',1)[0].isdigit()]

def make_pretagged(src, dst, chunk=32):
    bs=split_blocks(src.read_text(encoding='utf-8')); out=[]
    for start in range(0,len(bs),chunk):
        sub=bs[start:start+chunk]
        forms=[[r[1] for r in integer_rows(b)] for b in sub]
        doc=tagger(forms)
        assert len(doc.sentences)==len(sub)
        for block,sent,gold_forms in zip(sub,doc.sentences,forms):
            assert [w.text for w in sent.words] == gold_forms
            pred=iter(sent.words); lines=[]
            for line in block.splitlines():
                if line and not line.startswith('#') and line.split('\t',1)[0].isdigit():
                    c=line.split('\t'); w=next(pred)
                    c[2]=w.lemma or '_'; c[3]=w.upos or '_'; c[4]=w.xpos or '_'; c[5]=w.feats or '_'
                    line='\t'.join(c)
                lines.append(line)
            try: next(pred); raise AssertionError('extra predicted word')
            except StopIteration: pass
            out.append('\n'.join(lines))
    dst.write_text('\n\n'.join(out)+'\n',encoding='utf-8')
    assert len(split_blocks(dst.read_text()))==len(bs)

cache_hashes={}
for split in ('train','dev','test'):
    dst=DATA/f'{split}.predposlemma.conllu'
    make_pretagged(DATA/f'{split}.conllu',dst)
    cache_hashes[split]=sha256(dst)
    print(split,'frozen cache',cache_hashes[split])
del tagger; gc.collect(); torch.cuda.empty_cache()

EXPECTED_CACHE_SHA256={
 'train':'806b3d42c4f6838d6ba9c6565f29062a0d7b6e136e12d158e4449c3e7e970e89',
 'dev':'e88aa03ff7453469deda68ed30c29dd16769e9839d2010b36abf91faace3bd97',
 'test':'8c5ee2a043ceb7870d636d798c3c2863c9da82adebcac9dffd53670cadb26a7f'}
assert cache_hashes==EXPECTED_CACHE_SHA256,(cache_hashes,EXPECTED_CACHE_SHA256)
print('Frozen POS/lemma caches match the original experiments: OK')


In [ ]:
# Persist every completed condition to Drive so Colab interruption is recoverable.
from google.colab import drive
drive.mount('/content/drive')
PERSIST=Path('/content/drive/MyDrive/yue_lora_fullft_data_scaling')
PERSIST.mkdir(parents=True,exist_ok=True)

train_positions=SPLIT_POSITIONS['train']
train_index={position:index for index,position in enumerate(train_positions)}
full_gold_blocks=split_blocks((DATA/'train.conllu').read_text(encoding='utf-8'))
full_pred_blocks=split_blocks((DATA/'train.predposlemma.conllu').read_text(encoding='utf-8'))
assert len(full_gold_blocks)==len(full_pred_blocks)==803

for subset_id,meta in SUBSETS.items():
    subset_dir=DATA/subset_id; subset_dir.mkdir(exist_ok=True)
    indices=sorted(train_index[position] for position in meta['positions'])
    gold_text='\n\n'.join(full_gold_blocks[index] for index in indices)+'\n\n'
    pred_text='\n\n'.join(full_pred_blocks[index] for index in indices)+'\n'
    gold_path=subset_dir/'train.conllu'; pred_path=subset_dir/'train.predposlemma.conllu'
    gold_path.write_text(gold_text,encoding='utf-8')
    pred_path.write_text(pred_text,encoding='utf-8')
    assert sha256(gold_path)==meta['gold_sha256'],(subset_id,sha256(gold_path),meta['gold_sha256'])
    meta['predposlemma_sha256']=sha256(pred_path)
    assert len(split_blocks(gold_text))==meta['sentence_count']
    print(subset_id,meta['sentence_count'],meta['source_distribution'],meta['gold_sha256'])

(PERSIST/'subset_manifest.json').write_text(json.dumps(SUBSETS,ensure_ascii=False,indent=2),encoding='utf-8')


In [ ]:
# Run the predeclared 4 x 2 factorial experiment.
# Completed conditions are skipped.  An interrupted condition resumes from its latest DEV evaluation.
import time
import shutil
from peft import get_peft_model_state_dict
from stanza.models.common.bert_embedding import load_bert
from stanza.models.common.peft_config import build_peft_wrapper
from stanza.models.common.pretrain import Pretrain
from stanza.models.common.doc import HEAD,DEPREL
from stanza.models.common.vocab import VOCAB_PREFIX_SIZE
from stanza.models.depparse.data import DataLoader,InfiniteBatch
from stanza.models.depparse.trainer import GraphTrainer
from stanza.models.depparse.transition.model import SubtreeCombination
from stanza.models.depparse.utils import predict_dataset
from stanza.models.pos.vocab import MultiVocab
from stanza.utils.conll import CoNLL

base_path=Path(artifacts['depparse']['path'])
base_ckpt=torch.load(base_path,map_location='cpu',weights_only=True)
assert base_ckpt.get('model_type','graph')=='graph'
old_vocab=MultiVocab.load_state_dict(base_ckpt['vocab'])
old_units=list(old_vocab['deprel']._id2unit)
old_n=len(old_units)-VOCAB_PREFIX_SIZE

def one_dependency_path(model_type):
    dependencies=resources['zh-hans']['depparse']['gsdsimp_electra-large'].get('dependencies',[])
    names=[item['package'] for item in dependencies if item.get('model')==model_type]
    paths=[MODELS/'zh-hans'/model_type/f'{name}.pt' for name in names]
    paths=[path for path in paths if path.exists()]
    if len(paths)!=1: paths=list((MODELS/'zh-hans'/model_type).glob('*.pt'))
    assert len(paths)==1,f'Cannot resolve {model_type}: {paths}'
    return paths[0]

pretrain_obj=Pretrain(filename=str(one_dependency_path('pretrain'))) if base_ckpt['config'].get('pretrain') else None

def reset_seed():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

def predict_to_file(trainer,loader,path):
    trainer.model.eval()
    with torch.inference_mode(): predictions=predict_dataset(trainer,loader)
    loader.doc.set([HEAD,DEPREL],[item for sentence in predictions for item in sentence])
    path.write_text(f'{loader.doc:C}\n\n',encoding='utf-8')
    return path

def strict_las(gold_path,pred_path,exclude_punct=False):
    correct=total=0
    for gold_block,pred_block in zip(split_blocks(gold_path.read_text()),split_blocks(pred_path.read_text())):
        gold_rows=integer_rows(gold_block); pred_rows=integer_rows(pred_block)
        assert len(gold_rows)==len(pred_rows)
        for gold,pred in zip(gold_rows,pred_rows):
            assert gold[:2]==pred[:2]
            if exclude_punct and gold[3]=='PUNCT': continue
            total+=1; correct+=int(gold[6]==pred[6] and gold[7]==pred[7])
    return correct/total

def make_expanded_base(subset_id,condition_dir):
    train_path=DATA/subset_id/'train.conllu'
    train_labels=sorted({row[7] for block in split_blocks(train_path.read_text()) for row in integer_rows(block)})
    new_labels=[label for label in train_labels if label not in old_vocab['deprel']]
    expanded=copy.deepcopy(base_ckpt)
    dep_state=expanded['vocab']['deprel']
    dep_state['_id2unit']=old_units+new_labels
    dep_state['_unit2id']={unit:index for index,unit in enumerate(dep_state['_id2unit'])}
    new_n=old_n+len(new_labels)
    if new_labels:
        changed=[]
        for name,tensor in list(expanded['model'].items()):
            if name=='deprel.scorer.W_bilin.weight':
                value=tensor.new_zeros(tensor.shape[0],tensor.shape[1],new_n); value[:,:,:old_n]=tensor
                expanded['model'][name]=value; changed.append(name)
            elif name=='deprel.scorer.W_bilin.bias':
                value=tensor.new_zeros(new_n); value[:old_n]=tensor
                expanded['model'][name]=value; changed.append(name)
            elif name.startswith('deprel_linear.') and tensor.ndim in (1,2) and tensor.shape[0]==old_n:
                value=tensor.new_zeros((new_n,)+tuple(tensor.shape[1:])); value[:old_n]=tensor
                expanded['model'][name]=value; changed.append(name)
        assert changed
    args=copy.deepcopy(expanded['config'])
    args['charlm_forward_file']=str(one_dependency_path('forward_charlm').resolve())
    args['charlm_backward_file']=str(one_dependency_path('backward_charlm').resolve())
    expanded['config']=args; expanded['global_step']=0; expanded['last_best_step']=0; expanded['dev_score_history']=[]
    path=condition_dir/'expanded_base.pt'
    torch.save(expanded,path,_use_new_zipfile_serialization=False)
    del expanded
    return path,args,new_labels

def run_condition(subset_id,method):
    persistent=PERSIST/subset_id/method
    persistent.mkdir(parents=True,exist_ok=True)
    meta=SUBSETS[subset_id]
    signature_payload={'protocol_version':2,'subset_id':subset_id,'method':method,'gold_sha256':meta['gold_sha256'],
                       'pred_sha256':meta['predposlemma_sha256'],'seed':SEED,'cfg':CFG,
                       'hf_revision':HF_COMMIT,'base_sha256':sha256(base_path)}
    signature=hashlib.sha256(json.dumps(signature_payload,sort_keys=True).encode()).hexdigest()
    completed=persistent/'result.json'
    if completed.exists():
        result=json.loads(completed.read_text())
        assert result['experiment_signature']==signature,(completed,result.get('experiment_signature'),signature)
        print('SKIP verified completed condition:',subset_id,method)
        return result

    resume_state_path=persistent/'resume_state.json'

    def atomic_json(path,payload):
        tmp=path.with_suffix(path.suffix+'.tmp')
        tmp.write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding='utf-8')
        tmp.replace(path)

    def atomic_torch(path,payload):
        tmp=path.with_suffix(path.suffix+'.tmp')
        torch.save(payload,tmp,_use_new_zipfile_serialization=False)
        tmp.replace(path)

    reset_seed(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    condition_dir=OUT/f'{subset_id}_{method}'; condition_dir.mkdir(exist_ok=True)
    expanded_path,args,new_labels=make_expanded_base(subset_id,condition_dir)
    initial=torch.load(expanded_path,map_location='cpu',weights_only=True)
    if method=='lora_r8':
        args.update(use_peft=True,bert_finetune=True,lora_rank=CFG['rank'],lora_alpha=CFG['alpha'],
            lora_dropout=CFG['dropout'],lora_target_modules=CFG['targets'],lora_modules_to_save=[],
            optim='adamw',second_optim=None,lr=CFG['parser_lr'],
            bert_learning_rate=CFG['transformer_lr']/CFG['parser_lr'],bert_start_finetuning=0,
            bert_warmup_steps=0,weight_decay=0.0,bert_weight_decay=0.0,
            max_grad_norm=CFG['max_grad_norm'],batch_size=CFG['batch_size'],seed=SEED,
            enable_gradient_checkpointing=True,augment_nopunct=0.0)
        bert,_=load_bert(HF_REPO,enable_gradient_checkpointing=True)
        pefted=build_peft_wrapper(bert,args,logging.getLogger('stanza'),adapter_name='depparse')
        initial['bert_lora']=get_peft_model_state_dict(pefted,adapter_name='depparse')
        del bert,pefted
    elif method=='full_ft':
        for key in list(args):
            if key.startswith('lora_') or key in ('peft_name','bert_lora'): args.pop(key,None)
        args.update(use_peft=False,bert_finetune=True,optim='adamw',second_optim=None,
            lr=CFG['parser_lr'],bert_learning_rate=CFG['transformer_lr']/CFG['parser_lr'],
            bert_start_finetuning=0,bert_warmup_steps=0,bert_finetune_layers=None,
            weight_decay=0.0,bert_weight_decay=0.0,max_grad_norm=CFG['max_grad_norm'],
            batch_size=CFG['batch_size'],seed=SEED,enable_gradient_checkpointing=True,augment_nopunct=0.0)
        initial.pop('bert_lora',None)
    else: raise ValueError(method)
    initial['config']=args
    compat_path=condition_dir/'initial.pt'
    torch.save(initial,compat_path,_use_new_zipfile_serialization=False)
    del initial
    gc.collect(); torch.cuda.empty_cache()

    load_args=dict(args); load_args.pop('transition_subtree_combination',None)
    load_args['charlm_forward_file']=str(one_dependency_path('forward_charlm').resolve())
    load_args['charlm_backward_file']=str(one_dependency_path('backward_charlm').resolve())
    resume_state=json.loads(resume_state_path.read_text()) if resume_state_path.exists() else None
    if resume_state is not None:
        assert resume_state['experiment_signature']==signature,(resume_state_path,resume_state.get('experiment_signature'),signature)
        resume_model=persistent/resume_state['current_model']
        resume_optim=persistent/resume_state['optimizer_state']
        resume_rng=persistent/resume_state['rng_state']
        for required in (resume_model,resume_optim,resume_rng,persistent/resume_state['best_model']):
            assert required.exists() and required.stat().st_size>0,required
        trainer=GraphTrainer.load(str(resume_model),pretrain=pretrain_obj,args=load_args,device=DEVICE,reset_history=False)
    else:
        trainer=GraphTrainer.load(str(compat_path),pretrain=pretrain_obj,args=load_args,device=DEVICE,reset_history=True)
    assert isinstance(trainer.args['transition_subtree_combination'],SubtreeCombination)
    if method=='full_ft':
        assert 'bert_model' in trainer.model.unsaved_modules
        trainer.model.unsaved_modules.remove('bert_model')
        for parameter in trainer.model.bert_model.parameters(): parameter.requires_grad=True
        trainer._Trainer__init_optim()

    if resume_state is not None:
        saved_optim=torch.load(resume_optim,map_location='cpu',weights_only=True)
        for key,state in saved_optim['optimizer'].items(): trainer.optimizer[key].load_state_dict(state)
        for key,state in saved_optim['scheduler'].items(): trainer.scheduler[key].load_state_dict(state)

    # Confirm old parser tensors are exactly preserved only at fresh initialization.
    if resume_state is None:
        loaded=trainer.model.get_params(skip_modules=True)
        for name,old in base_ckpt['model'].items():
            got=loaded[name].detach().cpu()
            if name=='deprel.scorer.W_bilin.weight': assert torch.equal(got[:,:,:old_n],old)
            elif name=='deprel.scorer.W_bilin.bias' or (name.startswith('deprel_linear.') and old.ndim in (1,2) and old.shape[0]==old_n):
                assert torch.equal(got[:old_n],old)
            else: assert got.shape==old.shape and torch.equal(got,old),name
        del loaded,got,old

    trainable=sum(parameter.numel() for parameter in trainer.model.parameters() if parameter.requires_grad)
    transformer_trainable=sum(parameter.numel() for name,parameter in trainer.model.named_parameters()
                              if name.startswith('bert_model.') and parameter.requires_grad)
    reset_seed()
    train_doc=CoNLL.conll2doc(input_file=str(DATA/subset_id/'train.predposlemma.conllu'))
    dev_doc=CoNLL.conll2doc(input_file=str(DATA/'dev.predposlemma.conllu'))
    train_loader=DataLoader(train_doc,CFG['batch_size'],trainer.args,pretrain_obj,vocab=trainer.vocab,
                            evaluation=False,bert_tokenizer=trainer.model.bert_tokenizer)
    dev_loader=DataLoader(dev_doc,CFG['batch_size'],trainer.args,pretrain_obj,vocab=trainer.vocab,
                          evaluation=True,sort_during_eval=True,bert_tokenizer=trainer.model.bert_tokenizer)
    steps_per_epoch=len(train_loader)
    assert steps_per_epoch>0
    max_steps=CFG['max_epochs']*steps_per_epoch
    eval_interval=CFG['eval_every_epochs']*steps_per_epoch
    patience_steps=CFG['patience_epochs']*steps_per_epoch
    if resume_state is None:
        infinite=InfiniteBatch(train_loader)
        history=[]
        dev_prediction=condition_dir/'dev.step0000.conllu'
        predict_to_file(trainer,dev_loader,dev_prediction)
        best_score=conll18(DATA/'dev.conllu',dev_prediction)['LAS'].f1
        best_step=0; start_step=0; prior_training_seconds=0.0
    else:
        # Checkpoints are written only at complete epoch boundaries.  Restore RNG, then
        # reshuffle once: this is exactly what the uninterrupted InfiniteBatch would do
        # on its next call after exhausting the previous epoch.
        rng=torch.load(resume_rng,map_location='cpu',weights_only=False)
        random.setstate(rng['python']); np.random.set_state(rng['numpy']); torch.set_rng_state(rng['torch'])
        if torch.cuda.is_available(): torch.cuda.set_rng_state_all(rng['cuda'])
        train_loader.reshuffle(); infinite=InfiniteBatch(train_loader)
        history=resume_state['history']; best_score=resume_state['best_score']; best_step=resume_state['best_step']
        start_step=resume_state['step']; prior_training_seconds=resume_state['training_seconds']
        assert trainer.global_step==start_step,(trainer.global_step,start_step)
        print('RESUME',subset_id,method,'from epoch',start_step/steps_per_epoch)

    def save_resume(step,training_seconds):
        model_name=f'resume_model.step{step:06d}.pt'
        optim_name=f'resume_optim.step{step:06d}.pt'
        rng_name=f'resume_rng.step{step:06d}.pt'
        model_path=persistent/model_name
        model_tmp=model_path.with_suffix('.pt.tmp')
        trainer.save(str(model_tmp),save_optimizer=False)
        assert model_tmp.exists() and model_tmp.stat().st_size>0
        model_tmp.replace(model_path)
        atomic_torch(persistent/optim_name,{
            'optimizer':{key:value.state_dict() for key,value in trainer.optimizer.items()},
            'scheduler':{key:value.state_dict() for key,value in trainer.scheduler.items()}})
        atomic_torch(persistent/rng_name,{
            'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
            'cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []})
        previous_best=resume_state.get('best_model') if resume_state else None
        best_model=model_name if best_step==step else previous_best
        assert best_model is not None
        state={'experiment_signature':signature,'step':step,'best_step':best_step,'best_score':best_score,
               'best_model':best_model,'current_model':model_name,'optimizer_state':optim_name,
               'rng_state':rng_name,'training_seconds':training_seconds,'history':history}
        atomic_json(resume_state_path,state)
        # Keep only the current model and the DEV-best model; remove superseded recovery files.
        keep={model_name,best_model,optim_name,rng_name}
        for pattern in ('resume_model.step*.pt','resume_optim.step*.pt','resume_rng.step*.pt'):
            for old_path in persistent.glob(pattern):
                if old_path.name not in keep: old_path.unlink()
        return state

    if resume_state is None:
        resume_state=save_resume(0,0.0)
    start=time.monotonic(); last_loss=None
    loop_end=start_step if start_step-best_step>=patience_steps else max_steps
    for step in range(start_step+1,loop_end+1):
        loss,_=trainer.update(infinite.next_batch(),eval=False); last_loss=float(loss); trainer.global_step=step
        if step%eval_interval==0:
            dev_prediction=condition_dir/f'dev.step{step:06d}.conllu'
            predict_to_file(trainer,dev_loader,dev_prediction)
            score=conll18(DATA/'dev.conllu',dev_prediction)['LAS'].f1
            history.append({'step':step,'epoch':step/steps_per_epoch,'dev_conll18_las':score,
                            'loss':last_loss,'prediction_sha256':sha256(dev_prediction)})
            print(subset_id,method,'epoch',step/steps_per_epoch,'dev LAS',100*score)
            if score>best_score:
                best_score=score; best_step=step
            training_seconds=prior_training_seconds+time.monotonic()-start
            resume_state=save_resume(step,training_seconds)
            if step-best_step>=patience_steps: break
    training_seconds=prior_training_seconds+time.monotonic()-start
    peak_allocated=torch.cuda.max_memory_allocated()/2**30
    peak_reserved=torch.cuda.max_memory_reserved()/2**30
    best_path=persistent/resume_state['best_model']
    checkpoint_sha=sha256(best_path); checkpoint_bytes=best_path.stat().st_size

    # Reload the DEV-selected checkpoint before the single predeclared TEST evaluation.
    del trainer,train_loader,dev_loader,infinite,train_doc,dev_doc
    gc.collect(); torch.cuda.empty_cache()
    best_trainer=GraphTrainer.load(str(best_path),pretrain=pretrain_obj,args=load_args,device=DEVICE)
    for parameter in best_trainer.model.parameters(): parameter.requires_grad=False
    best_trainer.optimizer={}; best_trainer.scheduler={}
    dev_doc=CoNLL.conll2doc(input_file=str(DATA/'dev.predposlemma.conllu'))
    test_doc=CoNLL.conll2doc(input_file=str(DATA/'test.predposlemma.conllu'))
    dev_loader=DataLoader(dev_doc,CFG['batch_size'],best_trainer.args,pretrain_obj,vocab=best_trainer.vocab,
        evaluation=True,sort_during_eval=True,bert_tokenizer=best_trainer.model.bert_tokenizer)
    test_loader=DataLoader(test_doc,CFG['batch_size'],best_trainer.args,pretrain_obj,vocab=best_trainer.vocab,
        evaluation=True,sort_during_eval=True,bert_tokenizer=best_trainer.model.bert_tokenizer)
    dev_best=persistent/'dev.best.pred.conllu'; test_best=persistent/'test.best.pred.conllu'
    predict_to_file(best_trainer,dev_loader,dev_best); predict_to_file(best_trainer,test_loader,test_best)
    dev_score=conll18(DATA/'dev.conllu',dev_best)['LAS'].f1
    assert abs(dev_score-best_score)<1e-12,(dev_score,best_score)
    test_scores=conll18(DATA/'test.conllu',test_best)
    subset_labels=sorted({row[7] for block in split_blocks((DATA/subset_id/'train.conllu').read_text()) for row in integer_rows(block)})
    dev_labels=sorted({row[7] for block in split_blocks((DATA/'dev.conllu').read_text()) for row in integer_rows(block)})
    result={'experiment_signature':signature,'subset_id':subset_id,'proportion':meta['proportion'],
      'train_sentences':meta['sentence_count'],'method':method,'seed':SEED,
      'steps_per_epoch':steps_per_epoch,'best_step':best_step,'best_epoch':best_step/steps_per_epoch,
      'dev_conll18_las_percent':100*dev_score,'test_conll18_las_percent':100*test_scores['LAS'].f1,
      'test_uas_percent':100*test_scores['UAS'].f1,
      'test_strict_las_percent':100*strict_las(DATA/'test.conllu',test_best),
      'test_strict_las_no_punct_percent':100*strict_las(DATA/'test.conllu',test_best,True),
      'trainable_parameters':trainable,'transformer_trainable_parameters':transformer_trainable,
      'training_seconds':training_seconds,'peak_gpu_allocated_gib':peak_allocated,
      'peak_gpu_reserved_gib':peak_reserved,'checkpoint_bytes':checkpoint_bytes,
      'checkpoint_sha256':checkpoint_sha,'dev_prediction_sha256':sha256(dev_best),
      'test_prediction_sha256':sha256(test_best),'new_deprel_labels':new_labels,
      'dev_labels_unseen_in_subset':sorted(set(dev_labels)-set(subset_labels)),
      'test_was_previously_used_for_mandarin_model_family_selection':True}
    atomic_json(persistent/'history.json',history)
    atomic_json(completed,result)
    print(json.dumps(result,ensure_ascii=False,indent=2))

    del best_trainer,dev_loader,test_loader,dev_doc,test_doc
    gc.collect(); torch.cuda.empty_cache()
    for path in condition_dir.glob('*'):
        if path.is_file(): path.unlink()
    condition_dir.rmdir()
    # Result and predictions are the durable artifacts; large recovery checkpoints are temporary.
    for pattern in ('resume_model.step*.pt','resume_optim.step*.pt','resume_rng.step*.pt'):
        for path in persistent.glob(pattern): path.unlink()
    resume_state_path.unlink(missing_ok=True)
    return result

all_results=[]
for subset_id in SUBSETS:
    for method in METHODS:
        print('\n==========',subset_id,method,'==========')
        all_results.append(run_condition(subset_id,method))
        gc.collect(); torch.cuda.empty_cache()


In [ ]:
# Aggregate fixed-TEST learning curves and export a compact report.
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

all_results=[]
for subset_id in SUBSETS:
    for method in METHODS:
        result_path=PERSIST/subset_id/method/'result.json'
        assert result_path.exists(),result_path
        all_results.append(json.loads(result_path.read_text()))
df=pd.DataFrame(all_results).sort_values(['train_sentences','method'])
df.to_csv(PERSIST/'data_scaling_results.csv',index=False)

pivot=df.pivot(index='train_sentences',columns='method',values='test_conll18_las_percent')
pivot['full_minus_lora_las_points']=pivot['full_ft']-pivot['lora_r8']
pivot.to_csv(PERSIST/'test_las_comparison.csv')

fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for method,label,color in [('lora_r8','LoRA r=8','#2563eb'),('full_ft','Full FT','#dc2626')]:
    part=df[df.method==method]
    axes[0].plot(part.train_sentences,part.dev_conll18_las_percent,marker='o',label=label,color=color)
    axes[1].plot(part.train_sentences,part.test_conll18_las_percent,marker='o',label=label,color=color)
axes[0].set_title('DEV learning curve'); axes[1].set_title('Fixed TEST learning curve')
for ax in axes:
    ax.set_xlabel('Cantonese training sentences'); ax.set_ylabel('CoNLL-2018 LAS (%)')
    ax.grid(alpha=.25); ax.legend(frameon=False)
fig.suptitle('Cantonese dependency parsing: LoRA vs full fine-tuning')
fig.tight_layout(); fig.savefig(PERSIST/'data_scaling_curves.png',dpi=180)

summary={'design':{'proportions':[meta['proportion'] for meta in SUBSETS.values()],
 'methods':METHODS,'seed':SEED,'max_epochs':CFG['max_epochs'],
 'eval_every_epochs':CFG['eval_every_epochs'],'patience_epochs':CFG['patience_epochs'],
 'nested_grouped_source_balanced_subsets':True,'fixed_dev':True,'fixed_test':True},
 'results':all_results,
 'test_caveat':'The fixed custom test had already been used for Mandarin model-family selection.',
 'single_seed_caveat':'This Run-all default uses one seed; small differences require multi-seed confirmation.'}
(PERSIST/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
display(df,pivot)

report_dir=WORK/'final_report'; report_dir.mkdir(exist_ok=True)
for name in ('subset_manifest.json','data_scaling_results.csv','test_las_comparison.csv','data_scaling_curves.png','summary.json'):
    shutil.copy2(PERSIST/name,report_dir/name)
for subset_id in SUBSETS:
    for method in METHODS:
        source_dir=PERSIST/subset_id/method
        for name in ('result.json','history.json','dev.best.pred.conllu','test.best.pred.conllu'):
            shutil.copy2(source_dir/name,report_dir/f'{subset_id}_{method}_{name}')
archive=shutil.make_archive('/content/yue_lora_vs_fullft_data_scaling_results','zip',root_dir=report_dir)
print('report archive:',archive,sha256(Path(archive)))
files.download(archive)
